# Day 8: Snowflake Storage Integration & Dynamic Tables

This notebook uses DuckDB to simulate Snowflake's incremental query behavior and SQL transformation logic for Dynamic Tables.

In [ ]:
import duckdb

con = duckdb.connect(database=':memory:')
print("DuckDB in-memory connection established.")

In [ ]:
# Simulate Silver Table (Iceberg external table in Snowflake)
con.execute("""
CREATE TABLE condition_occurrence (
    person_id VARCHAR,
    condition_concept_id INT,
    condition_start_date DATE
)
""")

con.execute("""
INSERT INTO condition_occurrence VALUES
('P1', 32020, '2023-01-15'),
('P1', 32021, '2023-02-20'),
('P2', 32020, '2023-03-10')
""")

print("Silver Layer: condition_occurrence")
print(con.execute("SELECT * FROM condition_occurrence").df())

In [ ]:
# Simulate Dynamic Table Creation (Gold Layer)
# Since DuckDB doesn't have auto-refreshing Dynamic Tables, we use a View to simulate the continuous transform.
con.execute("""
CREATE VIEW dt_patient_condition_summary AS
SELECT
    person_id,
    COUNT(DISTINCT condition_concept_id) as unique_conditions,
    MAX(condition_start_date) as last_diagnosis_date
FROM condition_occurrence
GROUP BY person_id
""")

print("\nGold Layer: dt_patient_condition_summary (Initial State)")
print(con.execute("SELECT * FROM dt_patient_condition_summary ORDER BY person_id").df())

In [ ]:
# Simulate incoming data and Dynamic Table lag refresh
con.execute("""
INSERT INTO condition_occurrence VALUES
('P1', 32022, '2023-11-01'),
('P3', 32020, '2023-11-05')
""")

print("\nGold Layer: dt_patient_condition_summary (After New Data Arrival - Simulating Refresh)")
print(con.execute("SELECT * FROM dt_patient_condition_summary ORDER BY person_id").df())